In [1]:
import json
import pandas as pd
import numpy as np
import os
import glob

In [2]:
data_dir = '../data/final_dataset'
mc_files = ['certamen_mc.json', 'nle_other_questions_2015.json', 'nle_other_questions_2020.json', 'nle_other_questions_2025.json']
mc_files = [os.path.join(data_dir, f) for f in mc_files]

file_to_data = {}
for file in mc_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)
    
    print(base_name, len(file_to_data[base_name]))

certamen_mc.json 315
nle_other_questions_2015.json 171
nle_other_questions_2020.json 173
nle_other_questions_2025.json 130


In [3]:
files = glob.glob(os.path.join(data_dir, '*.json'))
question_files = [f for f in files if 'reading_comp_questions' in f]
passage_files = [f for f in files if 'reading_comp_passages' in f]

for file in question_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)
    print(base_name, len(file_to_data[base_name]))

nle_reading_comp_questions_2015.json 103
nle_reading_comp_questions_2020.json 103
nle_reading_comp_questions_2025.json 176


In [4]:
file_to_resp = {}
#model_name = 'llama3-turbo'
#model_name = 'qwq'
model_name = 'o3-mini'

model_resp_dir = f'../data/model_responses/{model_name}'

for file in file_to_data.keys():
    base_name = os.path.basename(file)
    file_to_resp[base_name] = {}
    with open(os.path.join(model_resp_dir, base_name), 'r') as f:
        file_to_resp[base_name] = json.load(f)
    print(base_name, len(file_to_resp[base_name]))



certamen_mc.json 315
nle_other_questions_2015.json 171
nle_other_questions_2020.json 173
nle_other_questions_2025.json 130
nle_reading_comp_questions_2015.json 103
nle_reading_comp_questions_2020.json 103
nle_reading_comp_questions_2025.json 176


In [17]:
def parse_resp(resp_text):
    '''look for the letter answer in the response text'''
    lines = resp_text.split('\n')
    # usually in last line
    line = lines[-1]
    line = line.replace('*', '')
    # look for "Answer: A"
    if 'Answer:' in line:
        try:
            letter =line.split('Answer: ')[1].strip()
        except Exception as e:
            print(resp_text)
            print('line:', line)
            print(e)
        # double check we only have a single letter
        if len(letter) != 1:
            print(f'Len letter > 1: {letter} as an answer')
            return letter[0]
        return letter
    
    
    else:
        # look thru prev lines in reverse
        for line in reversed(lines[:-1]):
            line = line.replace('*', '')
            if 'Answer:' in line:
                try:
                    letter =line.split('Answer: ')[1].strip()
                except Exception as e:
                    print(resp_text)
                    print('line:', line)
                    print(e)
                # double check we only have a single letter
                if len(letter) != 1:
                    print(f'Len letter > 1: {letter}')
                    return letter[0]
                return letter
            elif ':' in line:
                try:
                    letter =line.split(':')[1].strip()
                except Exception as e:
                    print(resp_text)
                    print('line:', line)
                    print(e)
                # double check we only have a single letter
                if len(letter) != 1:
                    print(f'Len letter > 1: {letter} as an answer')
                    return letter[0]
                return letter
        


        print('error parsing')
        print(resp_text)
        return '0'



In [12]:
# make dfs
question_dfs = []
for file in file_to_data.keys():
    questions = file_to_data[file]
    question_df = pd.DataFrame(questions)
    question_dfs.append(question_df)

In [7]:
question_dfs[0]

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers
0,NJCL-Certamen,1996,NJCL-Certamen_1996_9a_1,multiple_choice,vocabulary,unknown,english,latin,What Latin preposition is the root of “country”?,"[A: CONTRA, B: CUM, C: ULTRA]",[A: CONTRA]
1,NJCL-Certamen,1996,NJCL-Certamen_1996_11_0,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[A: sauce, B: saline, C: salad, D: sausage, E:...",[E: sautee]
2,NJCL-Certamen,1996,NJCL-Certamen_1996_15a_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[A: HORTATORY]
3,NJCL-Certamen,1996,NJCL-Certamen_1996_15b_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[B: DELIBERATIVE]
4,NJCL-Certamen,1996,NJCL-Certamen_1996_15c_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[C: OPTATIVE]
...,...,...,...,...,...,...,...,...,...,...,...
310,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_654,multiple_choice,grammar,advanced,english,english,Which of the following is NOT a use of the gen...,"[A: Objective, B: Subjective, C: Partitive, D:...",[D: Means]
311,NJCL-Certamen,2002,NJCL-Certamen_2002_2_63,multiple_choice,vocabulary,unknown,english,latin,Which of the following animals is described in...,"[A: delphīnus, B: gallus, C: orca, D: testūdō,...",[B: gallus]
312,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_838,multiple_choice,vocabulary,advanced,english,english,"Which of the following, if any, is NOT derived...","[A: lieutenant, B: location, C: allocate, D: l...",[E: none of the above]
313,NJCL-Certamen,2002,NJCL-Certamen_2002_10_71,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[A: sojourn, B: conjure, C: meridian, D: circa...",[B: conjure]


In [8]:
file_to_resp['certamen_mc.json']

{'NJCL-Certamen_1996_9a_1': 'The English word “country” originates from the Old French word “contree,” which in turn comes from a Vulgar Latin form derived from the preposition CONTRA, originally meaning “against” or “opposite.” This reflects the idea of land lying opposite or beyond a town.\n\nAnswer: A',
 'NJCL-Certamen_1996_11_0': 'The words sauce, saline, salad, and sausage all trace back to the Latin sal/salsus meaning “salt” or “salted.” In contrast, sautée (from French sauté, “jumped”) does not derive from that same Latin root.\n\nAnswer: E',
 'NJCL-Certamen_1996_15a_5': 'This sentence employs the subjunctive in a hortatory sense. In Latin, when the first-person plural subjunctive is used in an exhortatory context (especially with an interrogative tone as in this sentence), it is intended to suggest or propose an action including the speaker. Here, "Hōs latrōnēs interficiāmus?" effectively means "Shall we kill these thieves?" or "Let\'s kill these thieves?" which is characterist

In [13]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    answer_dict = file_to_resp[file]
    answer_col = []
    for (i, row) in question_df.iterrows():
        q_id = row['question_id']

        model_answer = answer_dict[q_id]
        answer_col.append(model_answer)

    question_df['raw_resp'] = answer_col

In [18]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    question_df['parsed_resp'] = question_df['raw_resp'].apply(parse_resp)
    question_dfs[i] = question_df
question_dfs[0]

Len letter > 1: D. as an answer
error parsing
Omnēs līberī translates as "all the children." The phrase ībunt is "they will go." The word mēcum is a contraction of mēc (me) + cum (with), meaning "with me." Therefore, the phrase means "All the children will go with me to the theatre." 

Correct answer: D
Len letter > 1: A. as an answer
Len letter > 1: Aelia is not able to see Galeria. as an answer


,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,raw_resp,parsed_resp
0,NJCL-Certamen,1996,NJCL-Certamen_1996_9a_1,multiple_choice,vocabulary,unknown,english,latin,What Latin preposition is the root of “country”?,"[A: CONTRA, B: CUM, C: ULTRA]",[A: CONTRA],The English word “country” originates from the...,A
1,NJCL-Certamen,1996,NJCL-Certamen_1996_11_0,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[A: sauce, B: saline, C: salad, D: sausage, E:...",[E: sautee],"The words sauce, saline, salad, and sausage al...",E
2,NJCL-Certamen,1996,NJCL-Certamen_1996_15a_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[A: HORTATORY],This sentence employs the subjunctive in a hor...,A
3,NJCL-Certamen,1996,NJCL-Certamen_1996_15b_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[B: DELIBERATIVE],"The subjunctive in ""Quid agam, iūdicēs? quō mē...",B
4,NJCL-Certamen,1996,NJCL-Certamen_1996_15c_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[C: OPTATIVE],"In the sentence ""Utinam mē mortuum vīdissēs,"" ...",C
...,...,...,...,...,...,...,...,...,...,...,...,...,...
310,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_654,multiple_choice,grammar,advanced,english,english,Which of the following is NOT a use of the gen...,"[A: Objective, B: Subjective, C: Partitive, D:...",[D: Means],The genitive case in Greek and Latin is used t...,B
311,NJCL-Certamen,2002,NJCL-Certamen_2002_2_63,multiple_choice,vocabulary,unknown,english,latin,Which of the following animals is described in...,"[A: delphīnus, B: gallus, C: orca, D: testūdō,...",[B: gallus],"The Latin sentence translates as, “It is the o...",B
312,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_838,multiple_choice,vocabulary,advanced,english,english,"Which of the following, if any, is NOT derived...","[A: lieutenant, B: location, C: allocate, D: l...",[E: none of the above],All four words trace back to the Latin root lo...,E
313,NJCL-Certamen,2002,NJCL-Certamen_2002_10_71,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[A: sojourn, B: conjure, C: meridian, D: circa...",[B: conjure],Most of the words here derive from the Latin w...,B


In [52]:
question_dfs[0]['parsed_resp'] = question_dfs[0]['raw_resp'].apply(parse_resp)
question_dfs[0]

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,raw_resp,parsed_resp
0,NJCL-Certamen,1996,NJCL-Certamen_1996_9a_1,multiple_choice,vocabulary,unknown,english,latin,What Latin preposition is the root of “country”?,"[A: CONTRA, B: CUM, C: ULTRA]",[A: CONTRA],"The Latin preposition that is the root of ""cou...",A
1,NJCL-Certamen,1996,NJCL-Certamen_1996_11_0,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[A: sauce, B: saline, C: salad, D: sausage, E:...",[E: sautee],To determine which of the given options is NOT...,E
2,NJCL-Certamen,1996,NJCL-Certamen_1996_15a_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[A: HORTATORY],To determine the use of the subjunctive in the...,A
3,NJCL-Certamen,1996,NJCL-Certamen_1996_15b_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[B: DELIBERATIVE],"The sentence ""Quid agam, iūdicēs? quō mē verta...",B
4,NJCL-Certamen,1996,NJCL-Certamen_1996_15c_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[C: OPTATIVE],"The sentence ""Utinam mē mortuum vīdissēs"" is a...",C
...,...,...,...,...,...,...,...,...,...,...,...,...,...
310,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_654,multiple_choice,grammar,advanced,english,english,Which of the following is NOT a use of the gen...,"[A: Objective, B: Subjective, C: Partitive, D:...",[D: Means],The genitive case is a grammatical case used i...,D
311,NJCL-Certamen,2002,NJCL-Certamen_2002_2_63,multiple_choice,vocabulary,unknown,english,latin,Which of the following animals is described in...,"[A: delphīnus, B: gallus, C: orca, D: testūdō,...",[B: gallus],To determine which animal is described in the ...,B
312,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_838,multiple_choice,vocabulary,advanced,english,english,"Which of the following, if any, is NOT derived...","[A: lieutenant, B: location, C: allocate, D: l...",[E: none of the above],To determine which of the given words is NOT d...,E
313,NJCL-Certamen,2002,NJCL-Certamen_2002_10_71,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[A: sojourn, B: conjure, C: meridian, D: circa...",[B: conjure],To determine which word is not derived from th...,B


In [19]:
def score_single_mc_question(correct_answers: list, model_answer: str):
    '''assumes multiple choice'''

    correct_answer = correct_answers[0][0] # first character should be the letter choice

    return int(correct_answer == model_answer[0])


In [20]:
question_df

,source_name,source_year,question_id,question_format,question_content,passage_id,difficulty,question_language,answer_language,question,multiple_choice_options,answers,correctness_logic,n_required,raw_resp,parsed_resp
0,NLE,2025,NLE_2025_ADVANCED_LATIN_POETRY_31,multiple_choice,reading_comprehension,NLE_2025_ADVANCED_LATIN_POETRY_0,advanced,english,english,"According to lines 1-2, what does the Pennsylv...","[A: stores away food carefully, B: eats a lot ...",[A: stores away food carefully],na,None,The passage describes the Pennsylvanian rustic...,A
1,NLE,2025,NLE_2025_ADVANCED_LATIN_POETRY_32,multiple_choice,reading_comprehension,NLE_2025_ADVANCED_LATIN_POETRY_0,advanced,english,english,What leads him to do this (lines 1-2)?,"[A: He is moving to a new farm., B: The cold w...",[B: The cold winter lies ahead.],na,None,The answer is B.\n\nExplanation:\nThe passage ...,B
2,NLE,2025,NLE_2025_ADVANCED_LATIN_POETRY_33,multiple_choice,reading_comprehension,NLE_2025_ADVANCED_LATIN_POETRY_0,advanced,english,english,"In line 4, longa vorabit hiems is an example o...","[A: pleonasm, B: synchysis, C: oxymoron, D: pe...",[D: personification],na,None,"In the phrase longa vorabit hiems (""the long w...",D
3,NLE,2025,NLE_2025_ADVANCED_LATIN_POETRY_34,multiple_choice,reading_comprehension,NLE_2025_ADVANCED_LATIN_POETRY_0,advanced,english,english,The metonymy Cerealibus...donis in line 5 refe...,"[A: farming equipment, B: dining utensils, C: ...",[D: harvested food],na,None,"The term ""cerealibus"" comes from Latin ""cereal...",D
4,NLE,2025,NLE_2025_ADVANCED_LATIN_POETRY_35,multiple_choice,reading_comprehension,NLE_2025_ADVANCED_LATIN_POETRY_0,advanced,english,english,What is the scansion of the first four feet of...,"[A: DDSS, B: DDDD, C: DSDS, D: SDDD]",[B: DDDD],na,None,We begin by recalling that in a “dactylic‐hexa...,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,NLE,2025,NLE_2025_INTRODUCTION_TO_LATIN_36,multiple_choice,reading_comprehension,NLE_2025_INTRODUCTION_TO_LATIN_1,beginner,english,english,"In line 8, what indicates the audience enjoys ...","[A: They rise and clap., B: They sit and stomp...",[A: They rise and clap.],na,None,"In line 8, the text tells us that Roscius ente...",A
172,NLE,2025,NLE_2025_INTRODUCTION_TO_LATIN_37,multiple_choice,reading_comprehension,NLE_2025_INTRODUCTION_TO_LATIN_1,beginner,english,english,"In lines 9-10, why is Aelia crying?","[A: Aelia is not able to see Galeria., B: Aeli...",[A: Aelia is not able to see Galeria.],na,None,The answer is A: Aelia is not able to see Gale...,A
173,NLE,2025,NLE_2025_INTRODUCTION_TO_LATIN_38,multiple_choice,reading_comprehension,NLE_2025_INTRODUCTION_TO_LATIN_1,beginner,english,english,"In lines 10-11, whom is Aelia able to see?","[A: the actors, B: Roscius, C: the spectators,...",[C: the spectators],na,None,We must look at lines 10–11. There Aelia is up...,C
174,NLE,2025,NLE_2025_INTRODUCTION_TO_LATIN_39,multiple_choice,reading_comprehension,NLE_2025_INTRODUCTION_TO_LATIN_1,beginner,english,english,How does the sailor solve Aelia's problem (lin...,"[A: He carries her to the front., B: He places...",[B: He places her onto the wall.],na,None,The sailor solves Aelia's problem by lifting h...,B


In [21]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    scores = []
    for (i, row) in question_df.iterrows():
        q_id = row['question_id']
        correct_answers = row['answers']
        model_answer = row['parsed_resp']
        scores.append(score_single_mc_question(correct_answers, model_answer)) 

    question_df['score'] = scores
question_dfs[0]




TypeError: 'NoneType' object is not subscriptable

In [29]:
acc_per_file = {}
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    accuracy = question_df['score'].mean()
    acc_per_file[file] = accuracy
acc_per_file

{'certamen_mc.json': 0.8031746031746032,
 'nle_other_questions_2015.json': 0.9590643274853801,
 'nle_other_questions_2020.json': 0.9595375722543352,
 'nle_other_questions_2025.json': 0.9461538461538461,
 'nle_reading_comp_questions_2015.json': 0.8932038834951457,
 'nle_reading_comp_questions_2020.json': 0.9029126213592233,
 'nle_reading_comp_questions_2025.json': 0.9829545454545454}

In [15]:
# incorrect answers 
certamen_df = question_dfs[0]
incorrect_df = certamen_df[certamen_df['score'] == 0]
incorrect_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,raw_resp,parsed_resp,score
6,NJCL-Certamen,1996,NJCL-Certamen_1996_12a_8,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[A: consummate, B: essence, C: present, D: ent...",[A: consummate],"To answer this question, let's break down each...",E,0
8,NJCL-Certamen,1996,NJCL-Certamen_1996_12c_7,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[A: habit, B: have, C: ability, D: prohibit, E...",[B: have],To determine which of the following is NOT der...,F,0
9,NJCL-Certamen,1996,NJCL-Certamen_1996_14a_9,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[A: location, B: couch, C: locomotion, D: locu...",[E: none of the above],To determine which of the given options is NOT...,B,0
19,NJCL-Certamen,1996,NJCL-Certamen_1996_1b_30,multiple_choice,grammar,unknown,english,english,What use of the dative case is found in the fo...,"[A: COMPOUND VERBS, B: AGENCY, C: REFERENCE]",[B: AGENCY],"The sentence ""Nōbīs omnibus amandī sunt aliī"" ...",C,0
24,NJCL-Certamen,1996,NJCL-Certamen_1996_15a_22,multiple_choice,grammar,unknown,english,english,What use of the genitive case is found in the ...,"[A: POSSESSION, B: PARTITIVE (GEN. OF THE WHOL...",[A: POSSESSION],"The sentence ""Maxima est potentia imperātōris""...",C,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_409,multiple_choice,grammar,advanced,english,latin,"Which of the following, if any, does NOT belon...","[A: sorōrī, B: diēbus, C: virī, D: aquae, E: d...",[C: virī],To determine which of the given options does n...,F,0
297,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_414,multiple_choice,grammar,advanced,english,latin,"Which of the following, if any, does NOT belon...","[A: cordī, B: nūbe, C: animālī, D: prīncipibus...",[A: cordī],To determine which of the following does not b...,F,0
300,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_439,multiple_choice,grammar,advanced,english,latin,"Which of the following, if any, does NOT belon...","[A: leō, B: poēta, C: Mārce, D: mHitī, E: amīc...",[D: mHitī],To determine which of the given words does not...,C,0
303,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_447,multiple_choice,vocabulary,advanced,english,latin,Which of the following does NOT belong because...,"[A: iānua, B: līmes, C: ostium, D: valvae, E: ...",[B: līmes],To determine which of the given Latin words do...,E,0


In [30]:
# create combined df, with new columns called "source file"
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    question_df['source_file'] = [file] * len(question_df)
    question_dfs[i] = question_df

combined_df = pd.concat(question_dfs)
combined_df



,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,raw_resp,parsed_resp,score,source_file,passage_id,correctness_logic,n_required
0,NJCL-Certamen,1996,NJCL-Certamen_1996_9a_1,multiple_choice,vocabulary,unknown,english,latin,What Latin preposition is the root of “country”?,"[A: CONTRA, B: CUM, C: ULTRA]",[A: CONTRA],"<think>\nOkay, the user is asking which Latin ...",A,1,certamen_mc.json,NaN,NaN,NaN
1,NJCL-Certamen,1996,NJCL-Certamen_1996_11_0,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[A: sauce, B: saline, C: salad, D: sausage, E:...",[E: sautee],"<think>\nOkay, so I have this question here as...",E,1,certamen_mc.json,NaN,NaN,NaN
2,NJCL-Certamen,1996,NJCL-Certamen_1996_15a_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[A: HORTATORY],"<think>\n\nOkay, let's see. The user is asking...",B,0,certamen_mc.json,NaN,NaN,NaN
3,NJCL-Certamen,1996,NJCL-Certamen_1996_15b_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[B: DELIBERATIVE],"<think>\n\nOkay, so I need to figure out what ...",B,1,certamen_mc.json,NaN,NaN,NaN
4,NJCL-Certamen,1996,NJCL-Certamen_1996_15c_5,multiple_choice,grammar,unknown,english,latin,What use of the subjunctive is found in this s...,"[A: HORTATORY, B: DELIBERATIVE, C: OPTATIVE]",[C: OPTATIVE],"<think>\n\nOkay, let's tackle this question. T...",C,1,certamen_mc.json,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,NLE,2025,NLE_2025_INTRODUCTION_TO_LATIN_36,multiple_choice,reading_comprehension,beginner,english,english,"In line 8, what indicates the audience enjoys ...","[A: They rise and clap., B: They sit and stomp...",[A: They rise and clap.],"<think>\nOkay, let me try to work through this...",A,1,nle_reading_comp_questions_2025.json,NLE_2025_INTRODUCTION_TO_LATIN_1,na,None
172,NLE,2025,NLE_2025_INTRODUCTION_TO_LATIN_37,multiple_choice,reading_comprehension,beginner,english,english,"In lines 9-10, why is Aelia crying?","[A: Aelia is not able to see Galeria., B: Aeli...",[A: Aelia is not able to see Galeria.],"<think>\n\nOkay, let's tackle this question. T...",A,1,nle_reading_comp_questions_2025.json,NLE_2025_INTRODUCTION_TO_LATIN_1,na,None
173,NLE,2025,NLE_2025_INTRODUCTION_TO_LATIN_38,multiple_choice,reading_comprehension,beginner,english,english,"In lines 10-11, whom is Aelia able to see?","[A: the actors, B: Roscius, C: the spectators,...",[C: the spectators],"<think>\nOkay, let's tackle this question. The...",C,1,nle_reading_comp_questions_2025.json,NLE_2025_INTRODUCTION_TO_LATIN_1,na,None
174,NLE,2025,NLE_2025_INTRODUCTION_TO_LATIN_39,multiple_choice,reading_comprehension,beginner,english,english,How does the sailor solve Aelia's problem (lin...,"[A: He carries her to the front., B: He places...",[B: He places her onto the wall.],"<think>\n\nOkay, let me try to figure out the ...",B,1,nle_reading_comp_questions_2025.json,NLE_2025_INTRODUCTION_TO_LATIN_1,na,None


In [31]:
# save combined df
save_dir = f'../data/model_responses_parsed/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# drop the columns question, multiple_choice_options, n_required, correctness_logic
save_df = combined_df.drop(columns=['question', 'multiple_choice_options', 'n_required', 'correctness_logic'])
save_dict = save_df.to_dict(orient='records')

# replace NaN for passage_id with empty string
for row in save_dict:
    if 'passage_id' in row and pd.isna(row['passage_id']):
        row['passage_id'] = ''

with open(os.path.join(save_dir, 'mc.json'), 'w') as f:
    json.dump(save_dict, f, indent=4)


In [60]:
question_df.groupby('difficulty')

In [61]:
def accuracy_by(df, group_cols, score_cols):
    """
    df          : DataFrame
    group_cols  : column or list of columns to group on
    score_cols  : str or list of str with binary-score columns
    """
    if isinstance(score_cols, str):
        score_cols = [score_cols]
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    # reshape wide → long so we can handle any number of *_score columns
    long = (
        df
        .melt(id_vars=group_cols,          # keep the grouping keys
              value_vars=score_cols,       # columns like 'bert_score', 'roberta_score', ...
              #var_name="model",            # model = original column name
              #value_name="score")   # 0/1 values
              )       
    )

    return (
        long
        .groupby(group_cols, dropna=False)
        .score
        .agg(accuracy="mean",  n_items="size")
        .reset_index()
    )

In [62]:
def accuracy_by(df, group_cols):
    """
    df          : DataFrame with a single 'score' column (0/1 values)
    group_cols  : column or list of columns to group on
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    return (
        df
        .groupby(group_cols, dropna=False)['score']
        .agg(accuracy='mean', n_items='size')
        .reset_index()
    )

In [63]:
acc_difficulty = accuracy_by(combined_df,
                             group_cols="difficulty",
                             #score_cols="score"
                             )
acc_difficulty

,difficulty,accuracy,n_items
0,advanced,0.910284,457
1,beginner,0.960145,276
2,intermediate,0.956897,232
3,unknown,0.786408,206


In [64]:
acc_lang_combos = accuracy_by(combined_df,
                             group_cols=["question_language", "answer_language"],
                             #score_cols=model_name+'_score'
                             )
acc_lang_combos

,question_language,answer_language,accuracy,n_items
0,english,english,0.919937,637
1,english,latin,0.862676,284
2,latin,english,0.953917,217
3,latin,latin,0.818182,33


In [65]:
# all latin-latin questions 
question_df[(question_df['question_language'] == 'latin') & (question_df['answer_language'] == 'latin')]

,source_name,source_year,question_id,question_format,question_content,passage_id,difficulty,question_language,answer_language,question,multiple_choice_options,answers,correctness_logic,n_required,raw_resp,parsed_resp,score,source_file


In [66]:
acc_content = accuracy_by(combined_df,
                             group_cols="question_content",
                             #score_cols=model_name+'_score'
                             )
acc_content

,question_content,accuracy,n_items
0,geography,0.846154,26
1,grammar,0.875000,312
2,history,1.000000,49
3,literary_devices,0.800000,10
4,literature,1.000000,16
5,mythology,0.974359,39
6,reading_comprehension,0.942857,385
7,scansion,0.000000,1
8,translation,0.981132,106
9,vocabulary,0.845815,227


In [67]:
acc_lang_content = accuracy_by(combined_df,
                             group_cols=["question_language", "answer_language", "question_content"],
                             #score_cols=model_name+'_score'
                             )
acc_lang_content

,question_language,answer_language,question_content,accuracy,n_items
0,english,english,geography,1.000000,6
1,english,english,grammar,0.912281,57
2,english,english,history,1.000000,31
3,english,english,literary_devices,0.714286,7
4,english,english,literature,1.000000,11
5,english,english,mythology,1.000000,25
6,english,english,reading_comprehension,0.948864,352
7,english,english,scansion,0.000000,1
8,english,english,translation,0.947368,19
9,english,english,vocabulary,0.812500,128


In [68]:
acc_year = accuracy_by(combined_df,
                        group_cols="source_year",
                        #score_cols=model_name+'_score'
                        )
acc_year

,source_year,accuracy,n_items
0,1996,0.796117,103
1,1999,1.000000,6
2,2000,0.825000,40
3,2002,0.777108,166
4,2015,0.956204,274
5,2020,0.960145,276
6,2025,0.941176,306


In [69]:
# for each content type, what proportions of languages are there? and vice versa
lang_combos = question_df[["question_language", "answer_language"]].value_counts()
content_types = question_df['question_content'].unique()

In [70]:
lang_combos

question_language  answer_language
english            english            159
                   latin                9
latin              english              8
Name: count, dtype: int64

In [71]:
content_types

array(['reading_comprehension', 'translation'], dtype=object)

In [72]:
lang_combos.index

MultiIndex([('english', 'english'),
            ('english',   'latin'),
            (  'latin', 'english')],
           names=['question_language', 'answer_language'])

In [73]:
pd.set_option('display.expand_frame_repr',False)

In [74]:
def add_missing_content_cols(df):
    cols = list(df.columns)
    new_df = df.copy()

    for content_type in content_types:
        if content_type not in cols:
            new_df[content_type] = [0]

    # sort cols
    sorted_types = sorted(list(content_types))
    new_df = new_df[list(sorted_types)]

    return new_df

In [75]:
sorted_types = sorted(list(content_types))
print('lang_combo', ' '.join(sorted_types))

for q_lang, a_lang in lang_combos.index:
    print(f"{q_lang}-{a_lang}", end=' ')
    # df with only this combo of langs 
    lang_df = question_df[(question_df['question_language'] == q_lang) &
                          (question_df['answer_language'] == a_lang)]
    
    # unique content types 
    proportions = lang_df['question_content'].value_counts()
    df = pd.DataFrame([proportions]) # transposes it
    df = add_missing_content_cols(df)
    print(' '.join([str(x) for x in df.iloc[0].values]))


lang_combo reading_comprehension translation
english-english 159 0
english-latin 9 0
latin-english 6 2


In [76]:
df.iloc[0].values

array([6, 2])

In [77]:
df.columns

Index(['reading_comprehension', 'translation'], dtype='object', name='question_content')

In [78]:
['grammar', 'vocabulary', 'mythology', 'geography'] == list(df.columns)

False